## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on the same modeling table, chooses the metrics and the threshold without touching the test set, and explains the models.

Sections: 4.1) shared evaluation function `train_eval()`, 4.2) draft data, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models, 4.6) shallow neural network, 4.7) hyperparameter tuning, 4.8) metric choice and threshold, 4.9) final model, 4.10) test set evaluation, 4.11) explainability (XAI).

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base = clean["model_base"]      # pre-race columns + the target `podium`
outcomes = clean["outcomes"]    # post-race columns (status, points...). Never features
tables = clean["tables"]        # all other cleaned tables

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (our models and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*, the standard summary of the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a chosen probability threshold. The threshold is 0.5 for now; it will be tuned on validation in a later step.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it for the ablation.

In [ ]:
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Draft data (temporary)

P3's modeling table is not ready yet (draft on Oct 8, frozen on Oct 10). To test the function now, we use `model_base` with the team's time split (train <= 2019, validation 2020-2021) and four pre-race columns. This cell will be replaced by P3's table; the test years (>= 2022) are not touched here.

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]

train = base[base["year"] <= 2019]
val = base[base["year"].between(2020, 2021)]

X_train, y_train = train[DRAFT_FEATURES].astype(float), train["podium"]
X_val, y_val = val[DRAFT_FEATURES].astype(float), val["podium"]

print(f"train: {len(X_train)} rows, podium rate {y_train.mean():.3f}")
print(f"validation: {len(X_val)} rows, podium rate {y_val.mean():.3f}")

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation) and **PR-AUC = the podium rate** (no better than random ranking), and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)")
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate and F1 = 0, exactly as expected, so the function computes the metrics correctly. Its accuracy is still about 85%, although it never finds a single podium. That is why we do not use accuracy, and why **PR-AUC is the natural primary metric** for this rare positive class. These dummy numbers are also the floor every real model must beat.

**Next step:** the baselines in section 4.4.

### 4.4 Baselines: a `grid <= 3` rule and logistic regression

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned. We use two:
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it only encodes the domain fact that the starting position is the strongest single predictor of a podium.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more in a reliable era").

Same draft data as section 4.2; the test years are not used.

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a yes/no score (a single operating point). `grid = 0` (pit-lane start, no grid slot) is not counted as a front-row start.

In [ ]:
import numpy as np

class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        p = X["grid"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3")
display(rule_results)

#### Attempt 2: logistic regression
`qual_position` is missing for most rows before the mid-1990s (no qualifying data, see Part 1), so the model needs an imputer. **Temporary preprocessing** until P3's pipeline is ready: fill missing values with the training median (`has_qualifying` already tells the model the value was missing), then standardize, so the weights are comparable and the solver converges. Both steps sit inside a `Pipeline`, so `train_eval()` fits them on the training years only.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(max_iter=1000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression")
display(lr_results)

#### Attempt 3: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training. We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)")
display(lr_bal_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results], ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned (standardized features, so the sizes are comparable)
coefs = pd.Series(lr.named_steps["logisticregression"].coef_[0], index=DRAFT_FEATURES)
display(coefs.sort_values().round(3).to_frame("weight (standardized)"))

**What we see (validation 2020-2021).**
* **Both baselines are far above the dummy** (PR-AUC 0.15), so the draft features carry real signal.
* **Logistic regression is the better ranker:** PR-AUC 0.667 vs 0.530 for the rule, and ROC-AUC 0.916 vs 0.820. On our primary metric (PR-AUC) it is the baseline to beat.
* **At threshold 0.5 the rule has the best F1** (0.695 vs 0.591). Plain logistic regression is too cautious at 0.5: precision 0.75 but recall only 0.49. This shows the threshold matters as much as the model, so it will be tuned on validation in a later step instead of staying at 0.5.
* **`class_weight="balanced"` behaved as expected:** PR-AUC and ROC-AUC barely change (0.666 vs 0.667), because the ranking stays the same; at 0.5 recall jumps to 0.91 and precision drops to 0.40. Weighting mostly moves the probabilities, which threshold tuning can also do.
* **Weights:** `grid` has by far the largest weight (-2.05: a lower grid number, i.e. a starting slot nearer the front, means a higher podium probability), then `qual_position` (-0.49). These are associations the model uses, not causes; `has_qualifying` and `pitlane_start` are also correlated with the era and with `grid`, so their individual signs should not be over-read.
* **Train scores are lower than validation** (PR-AUC 0.53 vs 0.67). This is not underfitting: the training years go back to 1950, when big fields and low reliability made the grid a weaker predictor. Era sensitivity is a known limitation, and the training start year is a decision for P3's split.

**Baseline to beat:** logistic regression, validation PR-AUC **0.667**. **Next step:** random forest and gradient boosting in section 4.5.


### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features. Both are in scikit-learn, so no new library is needed.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* large and slow-ish, and its probabilities are averages of votes, not calibrated.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. Usually the strongest model on tables like ours. *Limitation:* more hyperparameters, and it can overfit if it runs too long (scikit-learn stops early on a 10% slice of the training rows, never on our validation years).

**Class imbalance:** both use `class_weight="balanced"`, as required for the statistical models (podiums are about 14% of rows). Section 4.4 showed this mostly moves the probabilities; the threshold is tuned later.

Same draft data and temporary preprocessing as section 4.4; hyperparameters are tuned in a later step.

#### Attempt 4: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 races, which keeps the probabilities smooth instead of memorizing single rows. The median imputer is kept so the model gets exactly the same inputs as logistic regression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(SimpleImputer(strategy="median"),
                   RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                          n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest")
display(rf_results)

#### Attempt 5: gradient boosting
`HistGradientBoostingClassifier` handles missing values natively (it learns which side of each split the missing rows go to), so it needs no imputer and no scaling. Default settings for now.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting")
display(gb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap would mean overfitting).

In [ ]:
all_models = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results, rf_results, gb_results],
                       ignore_index=True)
pr = all_models.pivot(index="model", columns="split", values="PR-AUC")
summary = all_models[all_models["split"] == "validation"].set_index("model").drop(columns="split")
summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
display(summary.sort_values("PR-AUC", ascending=False))

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models do not beat logistic regression on the draft data.** PR-AUC: gradient boosting 0.6674, logistic regression 0.6665, random forest 0.6661; ROC-AUC is 0.915-0.917 for all three. Differences in the third decimal, with only about 117 podiums in the validation years, are a tie.
* **Why:** the draft table has only four features, and one of them, `grid`, carries almost all the signal in a simple shape (the nearer the front, the likelier the podium). A straight line already captures that, so the strengths of trees (non-linear effects, interactions) have nothing extra to learn. This is the brief's point that feature engineering matters as much as model choice: the comparison has to be repeated on P3's table, where form and standings features give the trees something to work with.
* **No overfitting:** the train-validation PR-AUC gap is about -0.14 for every model, the same era effect seen in section 4.4 (the training years go back to 1950). The forest (`min_samples_leaf=20`) and boosting (early stopping on training rows) did not memorize the training data.
* **At threshold 0.5** both balanced tree models behave like the balanced logistic regression: high recall (0.86-0.88), low precision (0.43-0.45). The threshold is tuned on validation in a later step.

**Decision:** keep random forest and gradient boosting as our two statistical models and re-run them on P3's table. The baseline to beat is still logistic regression. **Next step:** the shallow neural network in section 4.6.


### 4.6 Shallow neural network (PyTorch)

**How it works:** a feed-forward network. Each hidden layer computes weighted sums of its inputs and passes them through a ReLU (negative values become 0), which lets the network learn non-linear patterns. The output layer gives one score that the sigmoid turns into a podium probability. The weights are learned by gradient descent (Adam) on the binary cross-entropy loss. With no hidden layer this would be exactly logistic regression, so the hidden layers are what the network adds.

**Choices** ("shallow" = 1-2 hidden layers, as the brief asks):
* **Dropout 0.2:** during training, 20% of the hidden units are switched off at random in every batch, so the network cannot rely on single units (less overfitting).
* **Early stopping on validation:** after every epoch (one pass over the training rows) we compute the validation loss; training stops when it has not improved for 15 epochs, and the weights of the best epoch are kept.
* **Class imbalance:** `pos_weight` = (non-podium rows / podium rows) in the loss, the same idea as `class_weight="balanced"` for the other models.
* **Preprocessing:** a network needs numbers without gaps and on similar scales, so the same median imputer and scaler as logistic regression, fitted on the training rows only inside `fit()`.
* **Reproducible:** `torch.manual_seed(SEED)`, a seeded batch order and a single CPU thread (the number of threads changes the rounding of the sums), so Run All gives the same numbers every time, on a laptop and on Kaggle.

*Limitations:* many settings (layers, units, dropout, learning rate), needs scaling and imputation, it is a black box without XAI, and on small tables it rarely beats tree models. **Caveat:** the validation set chooses the stopping epoch, so the network's validation score is slightly optimistic compared with the other models; the unseen test set gives the fair comparison.

The class below only wraps the network with `fit` / `predict_proba`, so the same `train_eval()` scores it.

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


class ShallowNN:
    """Feed-forward network with 1-2 hidden layers, wrapped with fit / predict_proba so train_eval() can score it.
    - preprocessing (median imputer + scaler) is fitted on the training rows only, inside fit()
    - early stopping: training stops when the validation loss has not improved for `patience` epochs,
      and the weights of the best epoch are kept"""

    def __init__(self, X_val, y_val, hidden=(32,), dropout=0.2, lr=1e-3, batch_size=256,
                 max_epochs=200, patience=15, seed=SEED):
        self.X_val, self.y_val = X_val, y_val
        self.hidden, self.dropout, self.lr, self.batch_size = hidden, dropout, lr, batch_size
        self.max_epochs, self.patience, self.seed = max_epochs, patience, seed

    def _build(self, n_inputs):
        layers, size = [], n_inputs
        for h in self.hidden:                                   # hidden layer: linear -> ReLU -> dropout
            layers += [nn.Linear(size, h), nn.ReLU(), nn.Dropout(self.dropout)]
            size = h
        layers.append(nn.Linear(size, 1))                       # output: one score (logit) for "podium"
        return nn.Sequential(*layers)

    def _tensor(self, X):
        return torch.tensor(self.prep.transform(X), dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        torch.set_num_threads(1)                                # same numbers on every machine (thread count changes rounding)
        self.prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit(X)
        Xt, yt = self._tensor(X), torch.tensor(np.asarray(y), dtype=torch.float32)
        Xv, yv = self._tensor(self.X_val), torch.tensor(np.asarray(self.y_val), dtype=torch.float32)

        self.net = self._build(Xt.shape[1])
        pos_weight = (yt == 0).sum() / (yt == 1).sum()          # class imbalance, like class_weight="balanced"
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.lr)
        gen = torch.Generator().manual_seed(self.seed)

        self.history = {"train": [], "validation": []}
        best_loss, best_state, waited = float("inf"), None, 0
        for epoch in range(self.max_epochs):
            self.net.train()                                   # dropout on
            batch_losses = []
            for idx in torch.randperm(len(Xt), generator=gen).split(self.batch_size):
                optimizer.zero_grad()
                loss = loss_fn(self.net(Xt[idx]).squeeze(1), yt[idx])
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            self.net.eval()                                    # dropout off
            with torch.no_grad():
                val_loss = loss_fn(self.net(Xv).squeeze(1), yv).item()
            self.history["train"].append(float(np.mean(batch_losses)))
            self.history["validation"].append(val_loss)

            if val_loss < best_loss:                           # early stopping on the validation loss
                best_loss, waited = val_loss, 0
                best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                self.best_epoch = epoch + 1
            else:
                waited += 1
                if waited >= self.patience:
                    break
        self.net.load_state_dict(best_state)                   # keep the best epoch, not the last one
        return self

    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.net(self._tensor(X)).squeeze(1)).numpy()
        return np.column_stack([1 - p, p])

#### Attempt 6: one hidden layer (32 units)

In [ ]:
nn1 = ShallowNN(X_val, y_val, hidden=(32,))
nn1_results = train_eval(nn1, X_train, y_train, X_val, y_val, name="neural network (32)")
display(nn1_results)
print(f"stopped after {len(nn1.history['train'])} epochs, best epoch: {nn1.best_epoch}")

#### Attempt 7: two hidden layers (64 and 32 units)

In [ ]:
nn2 = ShallowNN(X_val, y_val, hidden=(64, 32))
nn2_results = train_eval(nn2, X_train, y_train, X_val, y_val, name="neural network (64-32)")
display(nn2_results)
print(f"stopped after {len(nn2.history['train'])} epochs, best epoch: {nn2.best_epoch}")

#### Loss curves
Training loss (average over the batches of each epoch, with dropout on) and validation loss per epoch. The dashed line marks the epoch whose weights were kept.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, net, title in [(axes[0], nn1, "1 hidden layer (32)"), (axes[1], nn2, "2 hidden layers (64-32)")]:
    epochs = range(1, len(net.history["train"]) + 1)
    ax.plot(epochs, net.history["train"], label="training loss")
    ax.plot(epochs, net.history["validation"], label="validation loss")
    ax.axvline(net.best_epoch, color="black", linestyle="--", label=f"best epoch ({net.best_epoch})")
    ax.set_title(f"Neural network, {title}")
    ax.set_xlabel("epoch")
    ax.legend()
axes[0].set_ylabel("weighted binary cross-entropy")
plt.tight_layout()
plt.show()

#### Comparison of all models so far (validation)

In [ ]:
all_models = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results, rf_results, gb_results,
                        nn1_results, nn2_results], ignore_index=True)
pr = all_models.pivot(index="model", columns="split", values="PR-AUC")
summary = all_models[all_models["split"] == "validation"].set_index("model").drop(columns="split")
summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
display(summary.sort_values("PR-AUC", ascending=False))

**What we see (validation 2020-2021).**
* **The networks score a little higher:** PR-AUC 0.6733 (two layers) and 0.6729 (one layer) vs 0.6665 for logistic regression, i.e. about +0.007. We do **not** call this a win, for two reasons: the validation set chose the stopping epoch (so the network's score is slightly optimistic), and with about 117 validation podiums a difference this small is within noise. On the draft data all learned models are practically tied, the same conclusion as section 4.5; the unseen test set will give the fair comparison.
* **Two layers vs one:** +0.0004 PR-AUC, practically identical for twice the depth. Unless P3's richer features change this, the simpler one-layer network is preferred.
* **Loss curves:** both losses fall quickly in the first ~10 epochs and then flatten; the validation loss never turns upward, so dropout and early stopping prevent overfitting, and training stops because the improvements become tiny. The validation loss is below the training loss because dropout is active while the training loss is measured, and because the training years include older, less predictable eras (as in section 4.4).
* **At threshold 0.5** the networks behave like the other balanced models: high recall (0.88), low precision (0.45). The PR-AUC train-validation gap (about -0.14) is also the same as for the other models.

**Next step:** tune the hyperparameters and choose the decision threshold on validation only (sections 4.7-4.9).


### 4.7 Hyperparameter tuning (validation only)

For each model we try a small grid of the settings that control its complexity. Every combination is trained on the training years and scored on validation with `train_eval()`; the best combination is the one with the highest **validation PR-AUC** (our primary metric, see 4.8). The test years are not touched.

The grids are deliberately small: the validation set has only about 117 podiums, so a large search would mostly fit its noise.

In [ ]:
from sklearn.model_selection import ParameterGrid

def tune(name, make_model, grid):
    """Train one model per combination in `grid` (training years) and score it on validation.
    Returns the fitted model with the best validation PR-AUC and the table of all results."""
    rows, best_model, best_pr = [], None, -1
    for params in ParameterGrid(grid):
        model = make_model(**params)
        res = train_eval(model, X_train, y_train, X_val, y_val, name=name).set_index("split")
        rows.append({**params, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"], "val ROC-AUC": res.loc["validation", "ROC-AUC"]})
        if res.loc["validation", "PR-AUC"] > best_pr:
            best_model, best_pr = model, res.loc["validation", "PR-AUC"]
    return best_model, pd.DataFrame(rows).sort_values("val PR-AUC", ascending=False, ignore_index=True)

tuned = {}   # model name -> (best fitted model, table of all combinations)

#### Attempt 8: tuning logistic regression
`C` is the inverse regularization strength: a small `C` pushes the weights towards 0 (simpler model), a large `C` lets them grow.

In [ ]:
tuned["logistic regression"] = tune(
    "logistic regression",
    lambda C: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                            LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    {"C": [0.01, 0.1, 1, 10]})
display(tuned["logistic regression"][1])

#### Attempt 9: tuning the random forest
`min_samples_leaf`: the minimum number of races behind each leaf (larger = smoother, less memorizing). `max_features`: how many features each split may look at (`"sqrt"` = a random subset, `1.0` = all of them; a subset makes the trees more different from each other).

In [ ]:
tuned["random forest"] = tune(
    "random forest",
    lambda min_samples_leaf, max_features: make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=min_samples_leaf, max_features=max_features,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)),
    {"min_samples_leaf": [5, 20, 50], "max_features": ["sqrt", 1.0]})
display(tuned["random forest"][1])

#### Attempt 10: tuning gradient boosting
`learning_rate`: how much each new tree may correct the previous ones (smaller = slower, more careful). `max_leaf_nodes`: the size of each tree (smaller = simpler trees).

In [ ]:
tuned["gradient boosting"] = tune(
    "gradient boosting",
    lambda learning_rate, max_leaf_nodes: HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, class_weight="balanced", random_state=SEED),
    {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]})
display(tuned["gradient boosting"][1])

#### Attempt 11: tuning the neural network
`hidden`: the size of the hidden layer(s). `dropout`: the share of hidden units switched off during training. (The number of epochs is already chosen by early stopping.)

In [ ]:
tuned["neural network"] = tune(
    "neural network",
    lambda hidden, dropout: ShallowNN(X_val, y_val, hidden=hidden, dropout=dropout),
    {"hidden": [(16,), (32,), (64, 32)], "dropout": [0.2, 0.4]})
display(tuned["neural network"][1])

**What we see (tuning, validation PR-AUC).**
* **Logistic regression:** `C` makes no difference (0.6665 for every value). With about 22,000 training rows and four features there is nothing to regularize.
* **Random forest:** best with `min_samples_leaf=5` and `max_features="sqrt"`: 0.6697 (0.6661 before tuning). Using all features with large leaves is clearly worse (0.6354).
* **Gradient boosting:** best with `learning_rate=0.1` and `max_leaf_nodes=15`: 0.6707 (0.6674 with the defaults).
* **Neural network:** best with two layers (64-32) and dropout 0.2: 0.6733, the setting already used in 4.6.
* **Tuning gains are tiny** (at most +0.004), much smaller than the noise of the validation PR-AUC measured in 4.9 (standard error about 0.046). Tuning cannot add information that the four draft features do not contain. The train PR-AUC barely moves either, so no combination overfits.


### 4.8 Metric choice and decision threshold

**Primary metric: PR-AUC.** Podiums are rare (about 15% of the validation rows). PR-AUC measures how well the real podium finishers are ranked at the top, it does not depend on a threshold, and its floor is the podium rate (0.15 for the dummy in 4.3) instead of 0.5, so it shows clearly how much a model adds.

**Secondary metric: F1 at the chosen threshold.** In the end the model must say yes or no; F1 balances precision (how many predicted podiums are right) and recall (how many real podiums are found).

ROC-AUC is reported for completeness, but with about 85% non-podium rows it looks good easily (0.82 even for the simple rule).

**Threshold:** for every tuned model we try thresholds from 0.05 to 0.95 (step 0.01) on validation and keep the one with the highest F1. Each model gets its own threshold, because class weighting shifts the probabilities (section 4.4).

In [ ]:
from sklearn.metrics import f1_score

def best_threshold(model, X, y):
    """Threshold between 0.05 and 0.95 (step 0.01) with the highest F1 on (X, y)."""
    proba = model.predict_proba(X)[:, 1]
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, proba >= t, zero_division=0) for t in candidates]
    return candidates[int(np.argmax(f1s))]

rows = []
for name, (model, _) in tuned.items():
    t = best_threshold(model, X_val, y_val)
    rows.append({"model": name, "threshold": t, **evaluate(model, X_val, y_val, threshold=t)})
tuned_table = pd.DataFrame(rows).set_index("model").round(4)
thresholds = tuned_table["threshold"].to_dict()
display(tuned_table)

### 4.9 Choosing the final model (validation only)

Before comparing the models we measure how noisy the validation PR-AUC is: we resample the validation rows with replacement 1000 times (bootstrap) and take the standard deviation of PR-AUC as its standard error (SE).

**Rule (one-standard-error rule):** among the models whose validation PR-AUC is within one SE of the best, take the simplest (order: logistic regression < random forest < gradient boosting < neural network). A difference smaller than the noise is not a reason to accept a more complex model. The final model and its threshold are then used once on the test set (next step) and by P1's inference function.

In [ ]:
from sklearn.metrics import average_precision_score

COMPLEXITY = ["logistic regression", "random forest", "gradient boosting", "neural network"]   # simplest first
val_pr = tuned_table["PR-AUC"]
best_name = val_pr.idxmax()

# bootstrap standard error of the best model's validation PR-AUC
rng = np.random.default_rng(SEED)
proba, y = tuned[best_name][0].predict_proba(X_val)[:, 1], y_val.to_numpy()
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    boot.append(average_precision_score(y[i], proba[i]))
se = np.std(boot)

FINAL_NAME = next(name for name in COMPLEXITY if val_pr[name] >= val_pr[best_name] - se)
FINAL_MODEL, FINAL_THRESHOLD = tuned[FINAL_NAME][0], thresholds[FINAL_NAME]

print(f"best validation PR-AUC: {best_name} = {val_pr[best_name]:.4f}, bootstrap SE = {se:.4f} "
      f"(95% range {np.percentile(boot, 2.5):.3f}-{np.percentile(boot, 97.5):.3f})")
print(f"models within one SE: {[n for n in COMPLEXITY if val_pr[n] >= val_pr[best_name] - se]}")
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}")

**How the final threshold was chosen:** precision, recall and F1 of the final model on validation for every threshold.

In [ ]:
proba = FINAL_MODEL.predict_proba(X_val)[:, 1]
candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([evaluate(FINAL_MODEL, X_val, y_val, threshold=t) for t in candidates], index=candidates)

ax = curve[["precision", "recall", "F1"]].plot(figsize=(8, 4))
ax.axvline(FINAL_THRESHOLD, color="black", linestyle="--", label=f"chosen threshold = {FINAL_THRESHOLD}")
ax.set_title(f"Final model ({FINAL_NAME}): metrics vs threshold on validation")
ax.set_xlabel("threshold (predict podium if probability >= threshold)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

# compare the final yes/no decisions with the simple rule grid <= 3 from section 4.4
rule_pred = X_val["grid"].between(1, 3).to_numpy()
disagree = int(((proba >= FINAL_THRESHOLD) != rule_pred).sum())
print(f"rows where the final model and the rule grid <= 3 disagree: {disagree} of {len(X_val)}")

**What we see (threshold and final model).**
* **Each model needs its own threshold:** 0.38 for logistic regression, 0.79 for the random forest, 0.82 for gradient boosting, 0.74 for the network. The three balanced models have their probabilities pushed up by the class weights, so their best threshold is higher, as expected from section 4.4.
* **The threshold mattered far more than the hyperparameters:** F1 rises from 0.55-0.60 at threshold 0.5 to about 0.70 for every model (logistic regression 0.6953, random forest and gradient boosting 0.6983, network 0.7011), a gain of about +0.10, against at most +0.004 PR-AUC from tuning.
* **The learned models rediscover the rule:** logistic regression, random forest and gradient boosting all end at precision about 0.70 and recall 0.69, the same operating point as the rule `grid <= 3` (F1 0.6953); the final model disagrees with the rule on only 2 of 774 validation rows. With these four features the best yes/no decision is essentially "the front three on the grid". The network picks a different trade-off: more recall (0.81), less precision (0.62).
* **Final model:** the bootstrap standard error of the validation PR-AUC is 0.046 (95% range 0.59-0.76), and all four models are within one standard error of the best, so the one-standard-error rule picks the simplest: **logistic regression with threshold 0.38**.
* **Threshold plot:** the curves move in steps because `grid` is a whole number; above about 0.66 the model never predicts a podium (its highest probability).

These choices are provisional: the same cells are re-run on P3's modeling table. The test set is used once, in the next step.

**Next step:** the comparison table on train / validation / test (test used once), ROC and PR curves, and confusion matrices (section 4.10).


### 4.10 Final evaluation on the test set (used once)

Until now the test years (2022-2024) were never used: every model, hyperparameter, threshold and the final choice were fixed on the training and validation years. Here each tuned model is scored **once** on the test set with the threshold it got on validation, next to its train and validation scores, so we can see whether the validation results hold on unseen seasons. Nothing is changed after this point. The rule `grid <= 3` is included as the reference to beat.

In [ ]:
test = base[base["year"] >= 2022]
X_test, y_test = test[DRAFT_FEATURES].astype(float), test["podium"]
print(f"test: {len(X_test)} rows, podium rate {y_test.mean():.3f} (= the PR-AUC of a model that knows nothing)")

final_models = {"rule grid <= 3": (GridRule(), 0.5),
                **{name: (model, thresholds[name]) for name, (model, _) in tuned.items()}}
rows = []
for name, (model, t) in final_models.items():
    label = f"{name} (final)" if name == FINAL_NAME else name
    for split, X, y in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
        rows.append({"model": label, "split": split, "threshold": t, **evaluate(model, X, y, threshold=t)})
comparison = pd.DataFrame(rows).round(4)
display(comparison)

# the two metrics we chose (4.8), side by side per split
display(comparison.pivot(index="model", columns="split", values=["PR-AUC", "F1"])
        .reindex(columns=["train", "validation", "test"], level=1))

#### ROC and precision-recall curves (test set)
The ROC curve shows the true-positive rate against the false-positive rate for every threshold; the precision-recall curve shows precision against recall. The area under each curve is the ROC-AUC and the PR-AUC (`AP`) in the table.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (model, _) in tuned.items():
    proba = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], color="black", linestyle="--", label="random guess (AUC = 0.5)")
ax_pr.axhline(y_test.mean(), color="black", linestyle="--", label=f"podium rate = {y_test.mean():.3f}")
ax_roc.set_title("ROC curves on the test set (2022-2024)")
ax_pr.set_title("Precision-recall curves on the test set (2022-2024)")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="lower left")
plt.tight_layout()
plt.show()

#### Confusion matrices (test set, threshold chosen on validation)
Rows are the real outcome, columns the prediction. Top-right = false alarms (predicted a podium that did not happen), bottom-left = missed podiums.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (model, _)) in zip(axes, tuned.items()):
    pred = (model.predict_proba(X_test)[:, 1] >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no podium", "podium"],
                                            ax=ax, colorbar=False)
    ax.set_title(f"{name}{' (final)' if name == FINAL_NAME else ''}\nthreshold {thresholds[name]}")
plt.tight_layout()
plt.show()

**What we see (test set 2022-2024, 1,351 rows, 204 podiums).**
* **The validation results hold on unseen seasons.** Test PR-AUC is 0.649-0.654 for all four models (0.666-0.673 on validation): a drop of at most 0.02, well inside the validation noise (standard error 0.046, section 4.9). Test ROC-AUC is 0.91-0.92. The choices made on validation did not overfit it, and the models carry over to a new technical era (the 2022 ground-effect cars).
* **Far above the references:** every learned model is about four times the no-skill floor (PR-AUC 0.151 = the test podium rate) and well above the rule `grid <= 3` (0.474). The rule's yes/no decisions are good, but it cannot rank drivers; the models can.
* **The tie between the models is confirmed:** gradient boosting and the network 0.6539, logistic regression 0.6530, random forest 0.6493. The order changed compared with validation and every difference is below 0.005, the ROC and PR curves lie on top of each other, and the confusion matrices of logistic regression, random forest and gradient boosting are almost identical. Choosing the simplest model (section 4.9) cost nothing.
* **Final model on the test set (logistic regression, threshold 0.38):** F1 0.645, precision 0.653, recall 0.637. It finds 130 of the 204 podiums, with 69 false alarms and 74 missed podiums, about the same yes/no decisions as the rule (F1 0.649).
* **The network has the highest test F1** (0.677: recall 0.77, precision 0.60) because on validation it picked a higher-recall trade-off. We do **not** switch to it after seeing the test results: picking a model on the test set would turn the test set into a second validation set, and its score would no longer be an honest estimate. The final model stays the one chosen on validation.
* **Train scores are lower than validation and test** (PR-AUC 0.53 vs 0.65-0.67) for every model: the training years go back to 1950, while validation and test are both modern seasons. This is the era sensitivity noted in section 4.4, not overfitting.

**Next step:** explain the final model (XAI) in section 4.11.


### 4.11 Explainability (XAI) of the final model

We explain the final model chosen in 4.9 (`FINAL_MODEL`). The code only calls `predict_proba`, so it works for any of our models. We use the validation years: explanations do not choose anything, so there is no reason to touch the test set again.

* **Permutation importance (global):** shuffle one feature at a time and measure how much the validation PR-AUC drops. A big drop means the model relies on that feature.
* **SHAP (global and local):** splits each prediction into one contribution per feature, starting from a baseline (the average prediction). The contributions add up exactly to the predicted probability.
* **LIME (local):** for one row, creates many slightly changed copies, asks the model about them, and fits a small linear model around that row. Its weights say which feature values push this one prediction up or down.

**Not causal.** All three methods describe how *the model* uses the features to make predictions. A feature being important is not proof that it causes podium finishes: starting at the front does not by itself put a driver on the podium; a fast car and driver cause both the good grid slot and the podium, and the model only uses the grid slot as a signal of that.

#### Global: permutation importance (validation, 30 shuffles per feature)

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

pr_auc_scorer = lambda model, X, y: average_precision_score(y, model.predict_proba(X)[:, 1])   # works for any of our models
pi = permutation_importance(FINAL_MODEL, X_val, y_val, scoring=pr_auc_scorer, n_repeats=30, random_state=SEED)
perm = pd.DataFrame({"PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=X_val.columns)
perm = perm.sort_values("PR-AUC drop")
display(perm[::-1].round(4))

ax = perm["PR-AUC drop"].plot.barh(xerr=perm["std"], figsize=(8, 3), color="tab:blue")
ax.set_title(f"Permutation importance of the final model ({FINAL_NAME}), validation")
ax.set_xlabel("drop in PR-AUC when the feature is shuffled (bar = mean, line = std)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

#### Global: SHAP summary (validation)
Each dot is one validation row. Position = how much that feature moved the predicted podium probability for that row (right = up, left = down); colour = the feature's value (red = high, blue = low). LIME (below) needs complete rows, so the single missing validation value is filled with the training median; the same reference rows are used for SHAP.

In [ ]:
import warnings
with warnings.catch_warnings():          # shap's import prints a harmless progress-bar warning in notebooks
    warnings.simplefilter("ignore")
    import shap

X_ref = X_val.fillna(X_train.median())
predict_podium = lambda X: FINAL_MODEL.predict_proba(pd.DataFrame(X, columns=X_val.columns))[:, 1]
background = X_ref.sample(100, random_state=SEED)   # SHAP's baseline = average prediction over these 100 rows
explainer = shap.Explainer(predict_podium, shap.maskers.Independent(background),
                           feature_names=list(X_val.columns), seed=SEED)
shap_values = explainer(X_ref, silent=True)
print(f"SHAP baseline (average predicted podium probability of the reference rows): {shap_values.base_values[0]:.3f}")

shap.plots.beeswarm(shap_values, show=False)
plt.title(f"SHAP summary of the final model ({FINAL_NAME}), validation")
plt.xlabel("SHAP value (change in predicted podium probability)")
plt.tight_layout()
plt.show()

#### Local: one podium row and one non-podium row
Chosen by a fixed rule, not by hand: the podium finisher with the **highest** predicted probability (a clear case), and the non-podium row with the **median** predicted probability among non-podiums (a typical midfield car). The finishing position is shown only to tell the story; it is post-race information and never a feature.

In [ ]:
races, drivers = tables["races"].set_index("raceId"), tables["drivers"].set_index("driverId")
proba, y = predict_podium(X_ref), y_val.to_numpy()
non_podium_rows = np.where(y == 0)[0]
examples = {
    "podium": int(np.argmax(np.where(y == 1, proba, -1))),
    "non-podium": int(non_podium_rows[np.argsort(proba[non_podium_rows])[len(non_podium_rows) // 2]]),
}

def describe(i):
    """Who/where for validation row i (finishing position from `outcomes`, for the story only)."""
    r = val.iloc[i]
    finished = outcomes.loc[val.index[i], "positionOrder"]
    return (f"{drivers.loc[r['driverId'], 'surname']}, {races.loc[r['raceId'], 'name']} {r['year']}: "
            f"grid {r['grid']}, predicted {proba[i]:.1%}, finished P{finished}")

for label, i in examples.items():
    print(f"{label:>10}: {describe(i)}")

**Local SHAP (waterfall):** starts at the baseline at the bottom and adds each feature's contribution until it reaches this row's predicted probability at the top.

In [ ]:
for label, i in examples.items():
    shap.plots.waterfall(shap_values[i], show=False)
    plt.title(f"Local SHAP, {label} example\n{describe(i)}")
    plt.tight_layout()
    plt.show()

**LIME:** the bars are the weights of LIME's small linear model around this row (green pushes towards podium, red away from it). We also print how close LIME's local model gets to the real model's probability.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

yes_no_columns = [j for j, c in enumerate(X_ref.columns) if X_ref[c].nunique() <= 2]   # treated as categories
lime_explainer = LimeTabularExplainer(X_ref.to_numpy(), feature_names=list(X_ref.columns),
                                      class_names=["no podium", "podium"], categorical_features=yes_no_columns,
                                      mode="classification", random_state=SEED)
predict_both = lambda X: FINAL_MODEL.predict_proba(pd.DataFrame(X, columns=X_ref.columns))

for label, i in examples.items():
    lime_result = lime_explainer.explain_instance(X_ref.iloc[i].to_numpy(), predict_both, num_features=X_ref.shape[1])
    lime_result.as_pyplot_figure()
    plt.title(f"LIME, {label} example\n{describe(i)}")
    plt.xlabel("LIME weight (effect on the podium probability)")
    plt.tight_layout()
    plt.show()
    print(f"{label}: model probability {proba[i]:.3f}, LIME's local estimate {lime_result.local_pred[0]:.3f}, "
          f"local fit R^2 {lime_result.score:.2f}")

#### Same inputs, different outcome
The non-podium row the model was most sure about, next to the podium example, and how often rows with exactly these inputs really finished on the podium.

In [ ]:
most_confident_miss = int(np.argmax(np.where(y == 0, proba, -1)))
print(f"podium example:      {describe(examples['podium'])}")
print(f"most confident miss: {describe(most_confident_miss)}")
same_inputs = (X_ref == X_ref.iloc[examples["podium"]]).all(axis=1).to_numpy()
print(f"validation rows with exactly these inputs: {same_inputs.sum()}, of which on the podium: "
      f"{y[same_inputs].sum()} ({y[same_inputs].mean():.0%}); the model predicts {proba[examples['podium']]:.0%} for each")

**Interpretation.**
* **The three methods agree on the order:** `grid` first, `qual_position` second, `pitlane_start` a distant third, `has_qualifying` last. Shuffling `grid` drops the validation PR-AUC by 0.36 and `qual_position` by 0.12 (permutation importance), and the SHAP summary shows the same ranking. This matches the logistic-regression weights in section 4.4 and the evidence of section 4.5: the starting position carries almost all the signal.
* **Direction (SHAP summary):** a low `grid` and `qual_position` (blue, near the front) push the probability up by up to about +0.3; high values (red, further back) push it down. A pit-lane start is rare but costs a lot (-0.2 to -0.35).
* **`has_qualifying` has zero importance here, which does not mean it is useless:** in 2020-2021 almost every row has qualifying data, so shuffling it changes nothing. It only matters for the old seasons in the training data. Importance is always relative to the data it is measured on.
* **Podium example (Bottas, Austria 2020, from pole):** SHAP starts at the baseline 0.146 and adds +0.32 for grid 1 and +0.17 for qualifying P1 to reach 0.651. **Non-podium example (Giovinazzi, Portugal 2021, grid 12):** grid 12 (-0.08) and qualifying 12th (-0.04) bring it down to 0.027. Both explanations are what a fan would expect.
* **LIME agrees on the direction but is only an approximation:** it ranks the same features first, but its small local model estimates 0.39 for the podium example (the real model says 0.65; local fit R^2 0.71) and fits the non-podium row poorly (R^2 0.15). SHAP values add up exactly to the model's prediction, so for this model we trust SHAP more for the size of each effect.
* **Same inputs, different outcome:** Hamilton at the 2020 Italian Grand Prix had exactly the same inputs as Bottas (grid 1, qualifying P1) and got the same 65%, but finished 7th. Of the 34 validation rows with these inputs, 27 (79%) reached the podium. What separates them happens during the race (incidents, penalties, strategy), which a pre-race model cannot see. The model also says 65% where 79% came true: it learned from 1950-2019, when a pole position converted to a podium less often (era sensitivity).

**Not causal.** These explanations describe the model, not Formula 1. The model relies on the grid position because it is the best pre-race summary of how fast the car and driver are; it does not show that starting position *causes* podiums, and moving a driver up the grid would not give him the model's probability.

**Next step:** model limitations write-up.
